# Train EGE-UNet on Colab, persist everything to Google Drive

This notebook trains EGE-UNet on a Colab GPU. All outputs (logs, checkpoints, TensorBoard events, `metrics.csv`, `test_results.json`) are written to `WORK_DIR` on **this session account's own Drive**, and the finished run is copied once into the **shared project folder** at the end.

**Key concepts:**
- The Drive `WORK_DIR` *is* the experiment: re-running this notebook with the same `WORK_DIR` automatically **resumes** from its `checkpoints/latest.pth`; a new `RUN_NAME` = a new experiment.
- **Never point `WORK_DIR` of parallel runs at the shared folder.** Several Colab sessions writing continuously into one Drive folder tree make DriveFS flaky (`torch.save: Parent directory does not exist`). Train into the session account's own Drive, then copy the finished run to the shared folder once (last cell).

**Prerequisites (one-time):**
1. The repo pushed to GitHub (`REPO_URL` below).
2. The shared project folder (owner account: `MyDrive/Viettel_AI_Race/EGE-UNet/`) containing `datasets/data_isic1718.zip` and `EGE-UNet-results/`, shared with the training accounts, each of which has a Drive shortcut so that `SHARED_ROOT` below resolves.

Then: **Runtime → Run all** (pick a GPU runtime first). When the Drive mount popup appears, **choose the account running this session** (not the owner account). If the session disconnects mid-training, run the notebook again from the top — training resumes where it left off. The final copy cell refuses to run until the training has actually finished.

In [ ]:
# ---------------- Parameters ----------------
REPO_URL   = 'https://github.com/thinhvd/EGE-UNet-exp.git'
DATASET    = 'isic17'                                            # 'isic17' | 'isic18'
HPA_MODE   = 'learnable'                                         # 'learnable' (original) | 'frozen_ones' | 'none'
GHPA_PLACEMENT = 'low'                                           # 'low' (original) | 'mid' | 'high'  — resolution band of the 6 GHPA modules
SEED       = 42                                                  # use 42, 43, 44 ... for multi-seed runs
PLACE_TAG  = '' if GHPA_PLACEMENT == 'low' else f'_{GHPA_PLACEMENT}'
RUN_NAME   = f'egeunet_{DATASET}_{HPA_MODE}{PLACE_TAG}_s{SEED}'  # new name => new experiment

# Training writes continuously HERE: the OWN Drive of the account running this session
# (single writer => no DriveFS concurrency problems; resume also lives here)
WORK_DIR   = f'/content/drive/MyDrive/EGE-UNet-results/{RUN_NAME}'

# The SHARED project folder (owner account: MyDrive/Viettel_AI_Race/EGE-UNet; training accounts
# reach it through a Drive shortcut). Only READ the zip at start + RECEIVE one copy at the end.
SHARED_ROOT = '/content/drive/MyDrive/Viettel_AI_Race/EGE-UNet'
DATA_ZIP    = f'{SHARED_ROOT}/datasets/data_isic1718.zip'

EXTRA_ARGS = ''                                                  # e.g. '--epochs 300 --num-workers 2'

# local (fast) copy of the dataset; the zip extracts to isic2017/ and isic2018/
DATA_DIR = '/content/data_isic1718/' + ('isic2017' if DATASET == 'isic17' else 'isic2018')

In [ ]:
# In the popup, choose the account RUNNING this session (its own Drive), not the owner account
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Sanity check: fail EARLY and LOUDLY if the Drive paths are wrong (case-sensitive!)
import os
assert os.path.isdir(SHARED_ROOT), (
    f'SHARED_ROOT not found: {SHARED_ROOT}\n'
    'Check the shortcut name/location with:  !ls /content/drive/MyDrive')
assert os.path.isfile(DATA_ZIP), f'DATA_ZIP not found: {DATA_ZIP}'
print('OK  SHARED_ROOT:', SHARED_ROOT)
print('OK  DATA_ZIP:   ', DATA_ZIP)
print('    WORK_DIR:   ', WORK_DIR)

In [ ]:
# Clone the repo on first run, pull updates on later runs
import os
if not os.path.isdir('/content/EGE-UNet'):
    !git clone {REPO_URL} /content/EGE-UNet
else:
    !git -C /content/EGE-UNet pull --ff-only
%cd /content/EGE-UNet

In [ ]:
# All requirements are preinstalled on Colab; this is a fast no-op kept for portability
!pip install -q -r requirements.txt

### Stage the dataset to the local disk

Drive is a network (FUSE) mount: opening each of the thousands of small image files is a network round-trip, which makes epochs **10–100× slower** if you read the dataset from Drive directly. Instead we copy **one** zip (full sequential bandwidth) and unzip it to the local SSD. This cell is skipped automatically if the data is already staged in this session.

In [ ]:
if not os.path.isdir(DATA_DIR):
    !cp "{DATA_ZIP}" /content/data_isic1718.zip
    !unzip -q /content/data_isic1718.zip -d /content/data_isic1718
    !rm /content/data_isic1718.zip
!ls {DATA_DIR}

### (Optional) Live TensorBoard

Started *before* training so the curves update live. It reads from Drive, so it can lag a little behind.

In [ ]:
%load_ext tensorboard
from tensorboard import notebook
notebook.start('--logdir "' + WORK_DIR + '/summary"')

### Train (or resume)

This is the cell to re-run after any disconnect. If a `latest.pth` already exists in `WORK_DIR/checkpoints/`, you will see a line like `resuming model from .../latest.pth. resume_epoch: N` and training continues from epoch N+1. Re-running an already-finished run is a harmless no-op.

Notes:
- GHPA ablation matrix: run this notebook once per `(HPA_MODE, SEED)` combination, e.g. `learnable / frozen_ones / none` × seeds `42, 43, 44` — the three modes can run **in parallel on three Colab accounts** since each writes to its own Drive. Analyse afterwards with `colab_analysis.ipynb` (owner account), after the copy cell below has moved the finished runs into the shared folder.
- `--num-workers 2` (via `EXTRA_ARGS`) speeds up data loading, but makes the augmentation RNG stream differ slightly from the paper's single-process setup.
- To intentionally restart from scratch in the same folder, add `--no-resume` to `EXTRA_ARGS` (or pick a new `RUN_NAME`).

In [ ]:
!python train.py --dataset {DATASET} --data-path "{DATA_DIR}" --work-dir "{WORK_DIR}" --hpa-mode {HPA_MODE} --ghpa-placement {GHPA_PLACEMENT} --seed {SEED} {EXTRA_ARGS}

### Inspect the artifacts stored on Drive

In [ ]:
import json
import pandas as pd

!find "{WORK_DIR}" -type f | sort

csv_path = f'{WORK_DIR}/metrics.csv'
if os.path.exists(csv_path):
    display(pd.read_csv(csv_path).tail(10))

results_path = f'{WORK_DIR}/test_results.json'
if os.path.exists(results_path):
    print(json.dumps(json.load(open(results_path)), indent=2))

### Copy the FINISHED run into the shared project folder

One bulk copy after training is safe (no concurrent-writer flakiness). The assert stops this cell while the run is unfinished — so "Run all" cannot copy a partial run. Re-running the cell is safe (it merges/overwrites the destination). After it succeeds, verify from the owner account that `Viettel_AI_Race/EGE-UNet/EGE-UNet-results/<RUN_NAME>/` contains `checkpoints/`, `metrics.csv`, `test_results.json`, `log/`.

In [ ]:
DEST = f'{SHARED_ROOT}/EGE-UNet-results'
assert os.path.isfile(f'{WORK_DIR}/test_results.json'), 'Run not finished yet (no test_results.json) - not copying'
!cp -r "{WORK_DIR}" "{DEST}/"
!du -sh "{WORK_DIR}" "{DEST}/{RUN_NAME}"
!ls "{DEST}/{RUN_NAME}"